In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv("learnpulse_dataset.csv")

print(df.head())
print(df.shape)

In [ ]:
print(df.isnull().sum())

In [ ]:
df.columns = [
    'user_id',
    'profile',
    'daily_study_hours',
    'focus_score',
    'completed_videos',
    'daily_breaks',
    'monthly_active_days',
    'streak_length',
    'consistency_index',
    'learning_efficiency',
    'productivity_score',
    'focus_growth',
    'is_burnout_risk',
    'course_completed',
    'efficiency_ratio',
    'is_break_overload',
    'engagement_score',
    'has_positive_momentum'
]

print(df.columns)

In [ ]:
before = df.shape[0]

df.drop_duplicates(inplace=True)

after = df.shape[0]

print("Duplicates Removed:", before - after)

In [ ]:
binary_cols = [
    'is_burnout_risk',
    'course_completed',
    'is_break_overload',
    'has_positive_momentum'
]

for col in binary_cols:
    df[col] = df[col].astype(int)

df.info()

In [ ]:
# Productivity per hour

df['productivity_per_hour'] = (
    df['productivity_score'] / df['study_hours']
)

# Video completion rate

df['video_completion_rate'] = (
    df['completed_videos'] / df['active_days']
)

# Healthy learning pattern

df['healthy_learning_pattern'] = np.where(
    (df['burnout_risk'] == 0) &
    (df['consistency_index'] > 70),
    1,
    0
)

df.head()

In [ ]:
print(df.columns)

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[
        (df[col] < lower) |
        (df[col] > upper)
    ]

    print(f"{col}: {len(outliers)} outliers")

In [ ]:
df = pd.get_dummies(
    df,
    columns=['profile'],
    drop_first=True
)

df.head()

In [ ]:
!pip install pymysql sqlalchemy

In [ ]:
from sqlalchemy import create_engine

# MySQL connection
username = "root"
password = "Pankaj"
host = "localhost"
port = "3306"
database = "learnpulse"

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")

# Write DataFrame to MySQL
table_name = "students"   # choose any table name
df.to_sql(table_name, engine, if_exists="replace", index=False)

# Read back sample
pd.read_sql("SELECT * FROM students LIMIT 5;", engine)